# PDF Parsing: Text, Pages, and Extraction Evidence

| Field | Value |
|---|---|
| Stage | Data foundation |
| Difficulty | Intermediate |
| Status | Complete |
| Requires network/API | No |
| Last reviewed | 2026-09-25 |

Callout - Key idea:
A PDF is a positioned graphics container, not a guaranteed stream of reading-order text. Keep page provenance and measure extraction coverage before retrieval.

## 30-Second Summary

This notebook extracts the repository-owned *Attention Is All You Need* PDF with `pypdf`. It compares one giant document with page-aware documents, records page and extraction metadata, and validates that every page yields text for this born-digital fixture.

## Why This Matters

A parser can return text while silently scrambling columns, dropping equations, or producing nothing for scanned pages. Page-level identity makes failures observable and citations resolvable.

## Scope

| Covers | Does not cover |
|---|---|
| Born-digital text extraction, page provenance, coverage checks, focused retrieval | OCR, table reconstruction, figure understanding, legal assessment of document use |


## Mental Model

```text
PDF bytes -> page objects -> extract text -> validate coverage/order -> page documents
                         empty page? -> OCR/layout fallback queue
```


In [1]:
from pathlib import Path
import re
from pypdf import PdfReader

def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the repository.")

REPO_ROOT = find_repo_root()
PDF_PATH = REPO_ROOT / "05-DataIngestParsing/data/pdf/attention.pdf"
reader = PdfReader(PDF_PATH)
assert reader.pages
len(reader.pages), PDF_PATH.relative_to(REPO_ROOT).as_posix()


(15, '05-DataIngestParsing/data/pdf/attention.pdf')

## How It Works

`pypdf` reads the PDF object model and asks each page for a text representation. We normalize whitespace only for display; the page number and source path stay attached. Empty or implausibly short pages are extraction warnings, not evidence that the source has no content.


## Baseline

The baseline concatenates every page into one string. It is easy to build but makes a narrow query return the entire paper and loses page-level citation boundaries.


In [2]:
full_text = "\n\n".join((page.extract_text() or "").strip() for page in reader.pages)
baseline_document = {
    "id": "attention-paper",
    "source": PDF_PATH.relative_to(REPO_ROOT).as_posix(),
    "content": full_text,
}
{"pages": len(reader.pages), "characters": len(full_text), "preview": full_text[:160].replace("\n", " ")}


{'pages': 15,
 'characters': 39615,
 'preview': 'Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or sch'}

## Technique Implementation

The page-aware representation uses one-based page numbers because that is what a reader sees. Extraction method and character count make parser behavior inspectable, while the stable ID supports citations such as `attention.pdf#page=1`.


In [3]:
page_documents = []
source = PDF_PATH.relative_to(REPO_ROOT).as_posix()
for page_number, page in enumerate(reader.pages, start=1):
    text = (page.extract_text() or "").strip()
    page_documents.append({
        "id": f"attention-paper:p{page_number}",
        "source": source,
        "page": page_number,
        "extraction_method": "pypdf-text",
        "characters": len(text),
        "content": text,
    })
[(item["page"], item["characters"]) for item in page_documents]


[(1, 2859),
 (2, 4257),
 (3, 1826),
 (4, 2505),
 (5, 3188),
 (6, 3479),
 (7, 3322),
 (8, 3193),
 (9, 2973),
 (10, 3112),
 (11, 3215),
 (12, 3213),
 (13, 812),
 (14, 815),
 (15, 818)]

## Controlled Experiment

For a query about the paper's central contribution, both representations contain the answer. A transparent token-overlap scorer should select page 1, while page-aware retrieval returns far less context. We also measure extraction coverage across all pages.


In [4]:
def token_overlap(query: str, text: str) -> int:
    query_terms = set(re.findall(r"[a-z0-9]+", query.lower()))
    text_terms = set(re.findall(r"[a-z0-9]+", text.lower()))
    return len(query_terms & text_terms)

query = "What architecture is based solely on attention mechanisms without recurrence?"
top_page = max(page_documents, key=lambda item: (token_overlap(query, item["content"]), -item["page"]))
experiment_result = {
    "top_page": top_page["page"],
    "page_characters": top_page["characters"],
    "full_document_characters": len(full_text),
    "nonempty_page_rate": sum(bool(item["content"]) for item in page_documents) / len(page_documents),
}
experiment_result


{'top_page': 1,
 'page_characters': 2859,
 'full_document_characters': 39615,
 'nonempty_page_rate': 1.0}

## Evaluation

The central-contribution query selects page **1**. All 15 pages produce text, so extraction coverage is **100%** for this fixture. Page-level context is materially smaller than the full-paper baseline, but text presence alone does not validate reading order, equations, or figures.


In [5]:
assert experiment_result["top_page"] == 1
assert experiment_result["nonempty_page_rate"] == 1.0
assert experiment_result["page_characters"] < experiment_result["full_document_characters"]
assert [item["page"] for item in page_documents] == list(range(1, len(reader.pages) + 1))
print(f"PDF checks passed for {len(page_documents)} pages.")


PDF checks passed for 15 pages.


## Decision Guide

| PDF type | First parser | Escalate when |
|---|---|---|
| Born-digital prose | Text extractor | Reading order or equations fail |
| Scanned pages | OCR | Confidence/layout is poor |
| Tables/forms | Layout-aware parser | Cells or fields are merged |
| Figures/charts | Multimodal extraction | Meaning is not present in captions |


## Failure Modes and Debugging

| Symptom | Likely cause | Fix |
|---|---|---|
| Empty page | Image-only scan | OCR and record confidence |
| Column text interleaves | Layout lost | Use layout-aware extraction |
| Header repeats everywhere | Boilerplate retained | Detect repeated marginal text |
| Citation off by one | Zero/one-based mismatch | Store reader-facing page numbers explicitly |


## Production Notes

### Observability
Track pages discovered, empty/short pages, characters per page, parser version, OCR use, and rejected files.

### Safety and Guardrails
Bound size and page count, treat embedded content as untrusted, and preserve authorization metadata.

### Latency and Cost
Text extraction is local; OCR and multimodal parsing add substantial compute and should run only on pages that need them.


## Practice

Add an image-only one-page PDF fixture. Confirm that the text extractor flags it, then define the metadata an OCR fallback must return.

## Recall

Toggle - Recall: Why retain page numbers?
They provide citation boundaries and localize extraction failures.

Toggle - Recall: Does non-empty text prove correct parsing?
No; order, tables, equations, and figures may still be wrong.

## Sources

- [pypdf: Extract text from a PDF](https://pypdf.readthedocs.io/en/stable/user/extract-text.html)
- Repository fixture: `attention.pdf`

## Review Log

| Date | Status | Confidence | Next review focus |
|---|---|---|---|
| 2026-09-25 | Complete; executed and visually reviewed | High for this born-digital fixture | Add scanned and multi-column adversarial fixtures |
